In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# MMFakeBench Evaluation

Evaluates your NewsCLIPpings-trained system on MMFakeBench val set.

**MMFakeBench categories:**
- `original` (300) - real image-text pairs
- `mismatch` (300) - image and text from different stories <- same as NewsCLIPpings
- `textual_veracity_distortion` (300) - text has been falsified
- `visual_veracity_distortion` (100) - image has been manipulated/AI-generated

**No retraining - zero-shot generalization test.**

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import clip as clip_lib
import warnings
warnings.filterwarnings("ignore")

# Paths
MM_ROOT        = _os.path.join(str(_cfg.MMFB_ROOT))
MM_VAL_JSON    = os.path.join(MM_ROOT, "MMFakeBench_val.json")
MM_VAL_IMAGES  = os.path.join(MM_ROOT, "MMFakeBench_val")
CLIP_MODEL_DIR = _os.path.join(str(_cfg.ROOT), 'models', 'clip')
CLIP_FEATURES  = _os.path.join(str(_cfg.ROOT), 'models', 'clip_finetuned_v2', 'val_features')
PROJECT_ROOT   = str(_cfg.ROOT)
AITR_WEIGHTS   = os.path.join(PROJECT_ROOT, "fusion_aitr", "aitr_weights.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


Device: cuda
GPU  : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM : 8.0 GB


In [2]:
# Load MMFakeBench val JSON
with open(MM_VAL_JSON, "r", encoding="utf-8") as f:
    mm_data = json.load(f)

mm_df = pd.DataFrame(mm_data)
mm_df["label"]      = (mm_df["gt_answers"] == "False").astype(int)  # 0=real, 1=fake
mm_df["image_full"] = mm_df["image_path"].apply(
    lambda p: os.path.join(MM_VAL_IMAGES, p.lstrip("/").lstrip("\\"))
)
mm_df["img_exists"] = mm_df["image_full"].apply(os.path.exists)

print(f"Total samples  : {len(mm_df)}")
print(f"Images found   : {mm_df['img_exists'].sum()}")
print(f"Images missing : {(~mm_df['img_exists']).sum()}")
print()
print("Label distribution:")
print(f"  Real (0): {(mm_df['label']==0).sum()}")
print(f"  Fake (1): {(mm_df['label']==1).sum()}")
print()
print("fake_cls distribution:")
print(mm_df["fake_cls"].value_counts().to_string())


Total samples  : 1000
Images found   : 1000
Images missing : 0

Label distribution:
  Real (0): 1000
  Fake (1): 0

fake_cls distribution:
fake_cls
original                       300
textual_veracity_distortion    300
mismatch                       300
visual_veracity_distortion     100


In [3]:
# Load CLIP
os.environ["CLIP_DOWNLOAD_ROOT"] = CLIP_MODEL_DIR
print("Loading CLIP ViT-L/14...")
clip_model, clip_preprocess = clip_lib.load("ViT-L/14", device=device, jit=False)
clip_model = clip_model.float().eval()
for p in clip_model.parameters():
    p.requires_grad = False
print("CLIP loaded.")

class MMFakeDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        print(f"Pre-tokenizing {len(df)} captions...")
        self.tokens = clip_lib.tokenize(
            [str(t)[:200] for t in df["text"].tolist()], truncate=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        try:
            img = Image.open(row["image_full"]).convert("RGB")
            img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return img, self.tokens[i], torch.tensor(row["label"], dtype=torch.long), i

val_transform = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.48145466, 0.4578275, 0.40821073],
                          [0.26862954, 0.26130258, 0.27577711]),
])

mm_dataset = MMFakeDataset(mm_df, val_transform)
mm_loader  = DataLoader(mm_dataset, batch_size=32, shuffle=False, num_workers=0)
print(f"Dataset ready: {len(mm_dataset)} samples")


Loading CLIP ViT-L/14...
CLIP loaded.
Pre-tokenizing 1000 captions...
Dataset ready: 1000 samples


In [4]:
# Extract CLIP features
print("Extracting CLIP features...")
all_img_feats, all_txt_feats = [], []
all_clip_sims, all_labels_mm = [], []

with torch.no_grad():
    for img, tokens, labels, idx in tqdm(mm_loader, desc="CLIP encode"):
        img    = img.to(device)
        tokens = tokens.to(device)
        img_feat = F.normalize(clip_model.encode_image(img).float(), dim=-1)
        txt_feat = F.normalize(clip_model.encode_text(tokens).float(), dim=-1)
        sims = (img_feat * txt_feat).sum(dim=-1)
        all_img_feats.append(img_feat.cpu())
        all_txt_feats.append(txt_feat.cpu())
        all_clip_sims.extend(sims.cpu().numpy())
        all_labels_mm.extend(labels.numpy())

mm_img_feats = torch.cat(all_img_feats, dim=0)
mm_txt_feats = torch.cat(all_txt_feats, dim=0)
mm_sims      = np.array(all_clip_sims)
mm_labels    = np.array(all_labels_mm)
print(f"Image features: {mm_img_feats.shape}")
print(f"Text  features: {mm_txt_feats.shape}")
print(f"CLIP sims range: {mm_sims.min():.3f} to {mm_sims.max():.3f}")


Extracting CLIP features...


CLIP encode: 100%|██████████| 32/32 [01:38<00:00,  3.07s/it]

Image features: torch.Size([1000, 768])
Text  features: torch.Size([1000, 768])
CLIP sims range: 0.046 to 0.402


In [5]:
# Load fine-tuned CLIP fusion head for classification probs
class CLIPFusion(nn.Module):
    def __init__(self, hidden_dim=512, dropout=0.5):
        super().__init__()
        embed_dim = 768
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 4, hidden_dim),
            nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, img_feat, txt_feat):
        fused = torch.cat([img_feat, txt_feat,
                           img_feat*txt_feat, img_feat-txt_feat], dim=-1)
        return self.fusion(fused).squeeze(-1)

v2_ckpt_path = _os.path.join(str(_cfg.ROOT), 'models', 'clip_finetuned_v2', 'clip_classifier.pt')
ckpt = torch.load(v2_ckpt_path, map_location=device, weights_only=False)

fusion_model = CLIPFusion().to(device)

# v2 stores weights under 'clip' prefix — extract fusion head only
model_state = ckpt['model_state']
print('All model state keys:')
for k in list(model_state.keys())[:15]:
    print(f'  {k}')

# Try different key prefixes
fusion_state = {}
for prefix in ['fusion.', 'clip.fusion.', 'model.fusion.']:
    fusion_state = {k.replace(prefix, ''): v
                    for k, v in model_state.items()
                    if k.startswith(prefix)}
    if fusion_state:
        print(f'Found fusion weights with prefix: {prefix}')
        break

if fusion_state:
    fusion_model.fusion.load_state_dict(fusion_state)
    fusion_model.eval()
    print(f'Loaded v2 fusion head from epoch {ckpt["epoch"]}')
    print(f'Best val acc: {max(ckpt["history"]["val_acc"]):.4f}')
else:
    print('WARNING: Could not find fusion weights — check key names above')

# Get clip_probs for all MM samples
mm_clip_probs = []
with torch.no_grad():
    for i in range(0, len(mm_img_feats), 64):
        img_b = mm_img_feats[i:i+64].to(device)
        txt_b = mm_txt_feats[i:i+64].to(device)
        probs = torch.sigmoid(fusion_model(img_b, txt_b)).cpu().numpy()
        mm_clip_probs.extend(probs)
mm_clip_probs = np.array(mm_clip_probs)
print(f'CLIP probs: min={mm_clip_probs.min():.3f} max={mm_clip_probs.max():.3f} mean={mm_clip_probs.mean():.3f}')

All model state keys:
  clip.positional_embedding
  clip.text_projection
  clip.logit_scale
  clip.visual.class_embedding
  clip.visual.positional_embedding
  clip.visual.proj
  clip.visual.conv1.weight
  clip.visual.ln_pre.weight
  clip.visual.ln_pre.bias
  clip.visual.transformer.resblocks.0.attn.in_proj_weight
  clip.visual.transformer.resblocks.0.attn.in_proj_bias
  clip.visual.transformer.resblocks.0.attn.out_proj.weight
  clip.visual.transformer.resblocks.0.attn.out_proj.bias
  clip.visual.transformer.resblocks.0.ln_1.weight
  clip.visual.transformer.resblocks.0.ln_1.bias
CLIP probs: min=0.139 max=0.840 mean=0.474


In [11]:
# AITR fusion
class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj    = nn.Sequential(nn.Linear(scalar_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        tokens   = torch.stack([img_emb, txt_emb, img_emb*txt_emb,
                                  img_emb-txt_emb, self.scalar_proj(scalar)], dim=1)
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        cls      = self.cls_token.expand(B, -1, -1)
        out      = self.transformer(torch.cat([cls, tokens], dim=1))
        return self.classifier(out[:, 0]).squeeze(-1)

# Scalar features with neutral placeholders for missing signals
n = len(mm_df)
mm_scalars = np.column_stack([
    mm_clip_probs,        # clip_prob
    mm_sims,              # clip_sim
    np.full(n, 0.33),     # deberta (neutral)
    np.zeros(n),          # s2 (no evidence available)
    np.zeros(n),          # s3
    np.zeros(n),          # s4
    np.zeros(n),          # s5
    np.zeros(n),          # s6
    np.full(n, 0.33),     # wiki (neutral)
])
scaler = StandardScaler()
mm_scalar_tensor = torch.tensor(scaler.fit_transform(mm_scalars), dtype=torch.float32)

aitr_model = AITR().to(device)
aitr_model.load_state_dict(torch.load(AITR_WEIGHTS, map_location=device))
aitr_model.eval()
print("AITR loaded.")

mm_ds = TensorDataset(mm_img_feats.to(device), mm_txt_feats.to(device), mm_scalar_tensor.to(device))
mm_ld = DataLoader(mm_ds, batch_size=64, shuffle=False, num_workers=0)

aitr_probs = []
with torch.no_grad():
    for img, txt, scl in tqdm(mm_ld, desc="AITR inference"):
        logits = aitr_model(img, txt, scl)
        aitr_probs.extend(torch.sigmoid(logits).cpu().numpy())
aitr_probs = np.array(aitr_probs)
print(f"AITR probs: min={aitr_probs.min():.3f} max={aitr_probs.max():.3f} mean={aitr_probs.mean():.3f}")


AITR loaded.


AITR inference: 100%|██████████| 16/16 [00:00<00:00, 61.02it/s]

AITR probs: min=0.003 max=0.999 mean=0.294


In [12]:
# Threshold sweep
print("Threshold sweep on MMFakeBench:")
best_acc, best_thr = 0, 0.54
for thr in np.arange(0.30, 0.76, 0.05):
    preds = (aitr_probs > thr).astype(int)
    acc   = accuracy_score(mm_labels, preds)
    f1    = f1_score(mm_labels, preds, zero_division=0)
    if acc > best_acc:
        best_acc, best_thr = acc, thr
    print(f"  thr={thr:.2f}  acc={acc*100:.2f}%  f1={f1:.4f}")

print(f"\nBest threshold: {best_thr:.2f} -> {best_acc*100:.2f}%")
mm_preds = (aitr_probs > best_thr).astype(int)

print()
print("=" * 65)
print("MMFAKEBENCH - OVERALL RESULTS")
print("=" * 65)
print(f"Accuracy : {accuracy_score(mm_labels, mm_preds)*100:.2f}%")
print(f"F1 (fake): {f1_score(mm_labels, mm_preds):.4f}")
print(f"AUC-ROC  : {roc_auc_score(mm_labels, aitr_probs):.4f}")
print()
print(classification_report(mm_labels, mm_preds, target_names=["REAL", "FAKE"]))

# Per category breakdown
print("=" * 65)
print("RESULTS BY CATEGORY")
print("=" * 65)
category_map = {
    "original"                   : "Real pairs - should predict REAL",
    "mismatch"                   : "Out-of-context <- trained on this",
    "textual_veracity_distortion": "False text - text signals weak",
    "visual_veracity_distortion" : "AI/edited images",
}
print(f'{"Category":<35} {"N":>5} {"Acc":>7} {"F1":>7} | Interpretation')
print("-" * 80)
for cat, interp in category_map.items():
    mask = mm_df["fake_cls"].values == cat
    if mask.sum() == 0:
        continue
    acc = accuracy_score(mm_labels[mask], mm_preds[mask])
    f1  = f1_score(mm_labels[mask], mm_preds[mask], zero_division=0)
    print(f"{cat:<35} {mask.sum():>5} {acc*100:>6.1f}% {f1:>7.4f} | {interp}")

# CLIP-only baseline
best_clip_acc, best_clip_thr = 0, 0.5
for thr in np.arange(0.30, 0.76, 0.02):
    acc = accuracy_score(mm_labels, (mm_clip_probs > thr).astype(int))
    if acc > best_clip_acc:
        best_clip_acc, best_clip_thr = acc, thr

print()
print("=" * 65)
print("CLIP-ONLY vs AITR")
print("=" * 65)
print(f"CLIP alone (thr={best_clip_thr:.2f}): {best_clip_acc*100:.2f}%")
print(f"AITR fusion (thr={best_thr:.2f})  : {best_acc*100:.2f}%")
print(f"Gain from fusion               : {(best_acc-best_clip_acc)*100:+.2f}%")

# Save results
results_summary = {
    "overall_accuracy": float(best_acc),
    "overall_f1"      : float(f1_score(mm_labels, mm_preds)),
    "overall_auc"     : float(roc_auc_score(mm_labels, aitr_probs)),
    "threshold"       : float(best_thr),
    "clip_only_acc"   : float(best_clip_acc),
    "per_category"    : {}
}
for cat in category_map:
    mask = mm_df["fake_cls"].values == cat
    if mask.sum() > 0:
        results_summary["per_category"][cat] = {
            "n"       : int(mask.sum()),
            "accuracy": float(accuracy_score(mm_labels[mask], mm_preds[mask])),
            "f1"      : float(f1_score(mm_labels[mask], mm_preds[mask], zero_division=0)),
        }
with open(os.path.join(PROJECT_ROOT, "mmfakebench_results.json"), "w") as f:
    json.dump(results_summary, f, indent=2)
print("\nResults saved to mmfakebench_results.json")


Threshold sweep on MMFakeBench:
  thr=0.30  acc=51.00%  f1=0.5149
  thr=0.35  acc=50.30%  f1=0.5005
  thr=0.40  acc=50.00%  f1=0.4939
  thr=0.45  acc=49.40%  f1=0.4847
  thr=0.50  acc=49.00%  f1=0.4753
  thr=0.55  acc=48.60%  f1=0.4679
  thr=0.60  acc=48.40%  f1=0.4636
  thr=0.65  acc=48.10%  f1=0.4577
  thr=0.70  acc=48.00%  f1=0.4526
  thr=0.75  acc=47.90%  f1=0.4498

Best threshold: 0.30 -> 51.00%

MMFAKEBENCH - OVERALL RESULTS
Accuracy : 51.00%
F1 (fake): 0.5149
AUC-ROC  : 0.6478

              precision    recall  f1-score   support

        REAL       0.36      0.83      0.51       300
        FAKE       0.84      0.37      0.51       700

    accuracy                           0.51      1000
   macro avg       0.60      0.60      0.51      1000
weighted avg       0.70      0.51      0.51      1000

RESULTS BY CATEGORY
Category                                N     Acc      F1 | Interpretation
--------------------------------------------------------------------------------
origina

In [8]:
# Per-subfolder breakdown
print("=" * 75)
print("PER-SUBFOLDER BREAKDOWN")
print("=" * 75)

mm_df["subfolder"] = mm_df["image_path"].apply(
    lambda p: p.strip("/").split("/")[1] if len(p.strip("/").split("/")) > 1 else "unknown"
)
mm_df["aitr_prob"] = aitr_probs
mm_df["aitr_pred"] = mm_preds
mm_df["correct"]   = (mm_df["aitr_pred"] == mm_df["label"]).astype(int)

subfolder_stats = mm_df.groupby("subfolder").agg(
    n        =("correct", "count"),
    accuracy =("correct", "mean"),
    avg_prob =("aitr_prob", "mean"),
).sort_values("accuracy", ascending=False)

print(f'{"Subfolder":<45} {"N":>4} {"Acc":>7} {"Avg prob":>9}')
print("-" * 70)
for subfolder, row in subfolder_stats.iterrows():
    print(f'{subfolder:<45} {int(row["n"]):>4} {row["accuracy"]*100:>6.1f}% {row["avg_prob"]:>9.3f}')


PER-SUBFOLDER BREAKDOWN
Subfolder                                        N     Acc  Avg prob
----------------------------------------------------------------------
llm_rewrite_val_30                              30   96.7%     0.064
bbc_val_50                                      50   94.0%     0.117
usa_today_val_50                                50   94.0%     0.084
wash_val_50                                     50   90.0%     0.140
coco_text_edit_val_50                           50   88.0%     0.178
fever_AI_val_100                               100   87.0%     0.179
guardian_val_50                                 50   86.0%     0.213
DGM4_text_edit_senti_val_50                     50   86.0%     0.170
coco_val_50                                     50   86.0%     0.192
fakeddit_val_50                                 50   82.0%     0.198
antifact_image_generation_val_50                50   78.0%     0.289
politicat_match_val_50                          50   74.0%     0.299
chatgpt_

In [9]:
print('Label distribution in mm_df:')
print(mm_df['gt_answers'].value_counts())
print()
print('Converted labels:')
print(f'  Real (0): {(mm_labels == 0).sum()}')
print(f'  Fake (1): {(mm_labels == 1).sum()}')
print()
print('AITR prob distribution:')
import numpy as np
print(f'  min={aitr_probs.min():.3f} max={aitr_probs.max():.3f} mean={aitr_probs.mean():.3f}')
print(f'  Probs > 0.5: {(aitr_probs > 0.5).sum()}')
print(f'  Probs > 0.3: {(aitr_probs > 0.3).sum()}')
print(f'  Probs > 0.1: {(aitr_probs > 0.1).sum()}')

Label distribution in mm_df:
gt_answers
Fake    700
True    300
Name: count, dtype: int64

Converted labels:
  Real (0): 1000
  Fake (1): 0

AITR prob distribution:
  min=0.003 max=0.999 mean=0.294
  Probs > 0.5: 272
  Probs > 0.3: 310
  Probs > 0.1: 383


In [10]:
# Fix label conversion
mm_df['label']  = (mm_df['gt_answers'] == 'Fake').astype(int)
mm_labels       = mm_df['label'].values

print('Fixed label distribution:')
print(f'  Real (0): {(mm_labels == 0).sum()}')
print(f'  Fake (1): {(mm_labels == 1).sum()}')

Fixed label distribution:
  Real (0): 300
  Fake (1): 700


In [13]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Fine-grained sweep
print('Fine-grained threshold sweep:')
best_acc, best_f1, best_thr_acc, best_thr_f1 = 0, 0, 0.3, 0.3
for thr in np.arange(0.05, 0.50, 0.01):
    preds = (aitr_probs > thr).astype(int)
    acc   = accuracy_score(mm_labels, preds)
    f1    = f1_score(mm_labels, preds, zero_division=0)
    if acc > best_acc:
        best_acc, best_thr_acc = acc, thr
    if f1 > best_f1:
        best_f1, best_thr_f1 = f1, thr

print(f'Best accuracy: {best_acc*100:.2f}% at threshold {best_thr_acc:.2f}')
print(f'Best F1      : {best_f1:.4f} at threshold {best_thr_f1:.2f}')

# Per category at best accuracy threshold
print()
print('Per category at best threshold:')
category_map = {
    'original'                   : 'Real pairs',
    'mismatch'                   : 'Out-of-context',
    'textual_veracity_distortion': 'False text',
    'visual_veracity_distortion' : 'AI/edited images',
}
mm_preds_best = (aitr_probs > best_thr_acc).astype(int)
for cat, desc in category_map.items():
    mask = mm_df['fake_cls'].values == cat
    if mask.sum() == 0:
        continue
    acc = accuracy_score(mm_labels[mask], mm_preds_best[mask])
    f1  = f1_score(mm_labels[mask], mm_preds_best[mask], zero_division=0)
    print(f'  {cat:<35} n={mask.sum():>3}  acc={acc*100:.1f}%  f1={f1:.4f}  | {desc}')

Fine-grained threshold sweep:
Best accuracy: 57.00% at threshold 0.05
Best F1      : 0.6208 at threshold 0.05

Per category at best threshold:
  original                            n=300  acc=72.7%  f1=0.0000  | Real pairs
  mismatch                            n=300  acc=59.0%  f1=0.7421  | Out-of-context
  textual_veracity_distortion         n=300  acc=40.0%  f1=0.5714  | False text
  visual_veracity_distortion          n=100  acc=55.0%  f1=0.7097  | AI/edited images


In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load Ateeq scores (200 samples: original + visual_veracity_distortion)
ateeq_df = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'features', 'mmfakebench_ai_scores_finetuned.csv'))
ateeq_df['true_label'] = (ateeq_df['gt_answers'] == 'Fake').astype(int)

# Merge AITR probs with Ateeq scores by image_path
mm_df['aitr_prob'] = aitr_probs
mm_df['true_label'] = mm_labels

merged = mm_df.merge(
    ateeq_df[['image_path', 'ai_score_ft', 'predicted_label_ft']],
    on='image_path', how='left'
)

print(f'Samples with Ateeq scores: {merged["ai_score_ft"].notna().sum()}/1000')
print(f'Samples without (mismatch + textual): {merged["ai_score_ft"].isna().sum()}/1000')

# Combined decision:
# If Ateeq score > 0.20 → FAKE (AI/manipulation detected)
# Else use AITR with threshold 0.05
ATEEQ_THR = 0.20
AITR_THR  = 0.05

combined_preds = []
for _, row in merged.iterrows():
    if pd.notna(row['ai_score_ft']) and row['ai_score_ft'] > ATEEQ_THR:
        combined_preds.append(1)  # Ateeq says fake
    elif row['aitr_prob'] > AITR_THR:
        combined_preds.append(1)  # AITR says fake
    else:
        combined_preds.append(0)  # Both say real
combined_preds = np.array(combined_preds)

print()
print('=' * 65)
print('COMBINED SYSTEM (AITR + Fine-tuned Ateeq)')
print('=' * 65)
print(f'Accuracy : {accuracy_score(mm_labels, combined_preds)*100:.2f}%')
print(f'F1       : {f1_score(mm_labels, combined_preds):.4f}')
print()
print(classification_report(mm_labels, combined_preds, target_names=['REAL', 'FAKE']))

print('Per category:')
category_map = {
    'original'                   : 'Real pairs',
    'mismatch'                   : 'Out-of-context',
    'textual_veracity_distortion': 'False text',
    'visual_veracity_distortion' : 'AI/edited images',
}
for cat, desc in category_map.items():
    mask = mm_df['fake_cls'].values == cat
    if mask.sum() == 0:
        continue
    acc = accuracy_score(mm_labels[mask], combined_preds[mask])
    f1  = f1_score(mm_labels[mask], combined_preds[mask], zero_division=0)
    print(f'  {cat:<35} n={mask.sum():>3}  acc={acc*100:.1f}%  f1={f1:.4f}  | {desc}')

print()
print('=' * 65)
print('COMPARISON')
print('=' * 65)
print(f'  AITR alone (thr=0.05)          : 57.00%')
print(f'  Ateeq alone (thr=0.20, 200 sam): 75.50%')
print(f'  Combined AITR + Ateeq          : {accuracy_score(mm_labels, combined_preds)*100:.2f}%')

Samples with Ateeq scores: 200/1000
Samples without (mismatch + textual): 800/1000

COMBINED SYSTEM (AITR + Fine-tuned Ateeq)
Accuracy : 55.70%
F1       : 0.6401

              precision    recall  f1-score   support

        REAL       0.35      0.54      0.42       300
        FAKE       0.74      0.56      0.64       700

    accuracy                           0.56      1000
   macro avg       0.54      0.55      0.53      1000
weighted avg       0.62      0.56      0.58      1000

Per category:
  original                            n=300  acc=54.3%  f1=0.0000  | Real pairs
  mismatch                            n=300  acc=59.0%  f1=0.7421  | Out-of-context
  textual_veracity_distortion         n=300  acc=40.0%  f1=0.5714  | False text
  visual_veracity_distortion          n=100  acc=97.0%  f1=0.9848  | AI/edited images

COMPARISON
  AITR alone (thr=0.05)          : 57.00%
  Ateeq alone (thr=0.20, 200 sam): 75.50%
  Combined AITR + Ateeq          : 55.70%


In [15]:
# Find optimal Ateeq threshold that maximizes overall accuracy
print('Ateeq threshold sweep on combined system:')
best_acc, best_thr = 0, 0.20
for ateeq_thr in np.arange(0.10, 0.95, 0.05):
    preds = []
    for _, row in merged.iterrows():
        if pd.notna(row['ai_score_ft']) and row['ai_score_ft'] > ateeq_thr:
            preds.append(1)
        elif row['aitr_prob'] > 0.05:
            preds.append(1)
        else:
            preds.append(0)
    preds = np.array(preds)
    acc = accuracy_score(mm_labels, preds)
    f1  = f1_score(mm_labels, preds, zero_division=0)
    if acc > best_acc:
        best_acc, best_thr = acc, ateeq_thr

    # Per category
    vv_mask = mm_df['fake_cls'].values == 'visual_veracity_distortion'
    orig_mask = mm_df['fake_cls'].values == 'original'
    vv_acc   = accuracy_score(mm_labels[vv_mask], preds[vv_mask])
    orig_acc = accuracy_score(mm_labels[orig_mask], preds[orig_mask])
    print(f'  ateeq_thr={ateeq_thr:.2f}  overall={acc*100:.1f}%  '
          f'original={orig_acc*100:.1f}%  visual_vd={vv_acc*100:.1f}%')

print(f'\nBest overall: {best_acc*100:.2f}% at Ateeq threshold {best_thr:.2f}')

Ateeq threshold sweep on combined system:
  ateeq_thr=0.10  overall=55.3%  original=52.0%  visual_vd=100.0%
  ateeq_thr=0.15  overall=55.3%  original=53.0%  visual_vd=97.0%
  ateeq_thr=0.20  overall=55.7%  original=54.3%  visual_vd=97.0%
  ateeq_thr=0.25  overall=56.1%  original=55.7%  visual_vd=97.0%
  ateeq_thr=0.30  overall=57.0%  original=59.0%  visual_vd=96.0%
  ateeq_thr=0.35  overall=57.8%  original=61.7%  visual_vd=96.0%
  ateeq_thr=0.40  overall=58.0%  original=62.3%  visual_vd=96.0%
  ateeq_thr=0.45  overall=58.0%  original=63.0%  visual_vd=94.0%
  ateeq_thr=0.50  overall=58.5%  original=65.0%  visual_vd=93.0%
  ateeq_thr=0.55  overall=59.0%  original=67.0%  visual_vd=92.0%
  ateeq_thr=0.60  overall=59.2%  original=68.0%  visual_vd=91.0%
  ateeq_thr=0.65  overall=59.3%  original=68.3%  visual_vd=91.0%
  ateeq_thr=0.70  overall=59.3%  original=69.3%  visual_vd=88.0%
  ateeq_thr=0.75  overall=59.4%  original=70.0%  visual_vd=87.0%
  ateeq_thr=0.80  overall=59.3%  original=70.0%